# Ropedia Academy — B7 · 3D Gaussian Splatting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B7.ipynb)

> **Splats explicit 3D Gaussians to 2D and renders them as alpha-blended ellipses (size~scale, alpha~opacity) — the rasterizer that makes 3DGS real-time.**
>
> 把显式 3D 高斯泼溅到 2D，并以 alpha 混合的椭圆渲染（大小~尺度，透明度~不透明度）——这正是让 3DGS 实时的光栅化器。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B7

In [ ]:
import torch, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# A scene = many explicit 3D GAUSSIANS (position, scale, color, opacity, + a label).
torch.manual_seed(1); N = 12
means   = torch.rand(N, 3) * torch.tensor([4.,4.,3.]) + torch.tensor([-2,-2,4.])
scales  = torch.rand(N) * 30 + 8
colors  = torch.rand(N, 3); opacity = torch.rand(N)*0.6 + 0.4
K = torch.tensor([[400,0,128.],[0,400,128.],[0,0,1.]])
uv = torch.stack([(K @ m)[:2] / (K @ m)[2] for m in means])      # splat centers -> 2D

# Render by FRONT-TO-BACK alpha blending (the rasterizer core).
pixel, Tr = torch.zeros(3), 1.0
for i in torch.argsort(means[:,2]):
    a = opacity[i].item(); pixel = pixel + Tr*a*colors[i]; Tr = Tr*(1-a)
print("rasterized center pixel:", pixel.round(decimals=2).tolist())

# Visualize the splatted 2D Gaussians (size ~ scale, alpha ~ opacity).
fig, ax = plt.subplots(figsize=(4.5,4.5))
for i in torch.argsort(-means[:,2]):                              # far -> near
    ax.add_patch(Ellipse(uv[i].tolist(), scales[i].item(), scales[i].item()*0.7,
                 color=colors[i].tolist(), alpha=opacity[i].item()))
ax.set_xlim(0,256); ax.set_ylim(256,0); ax.set_title("3D Gaussians splatted to 2D"); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B7
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks